In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations

from babel.util import distinct

In [ ]:
master_regulators = pd.read_csv("regulons_TFs_masters.txt", sep="\t")
tissues = sorted(master_regulators["tissue"].unique())

ppi = pd.read_csv("BIOGRID-PROJECT-alzheimers_disease_project-INTERACTIONS-5.0.252.tab3.txt", sep="\t")

gwas_genes = [
    "ICA1","CR1","CR1-AS1","ADAM17","YWHAQ","JAZF1","NCK2",
    "SEC61G-DT","EGFR","PMS2P1","EPHA1-AS1","CTSB","PTK2B",
    "CLU","SHARPIN","ABCA1","USP6NL-AS1","ECHDC3",
    "LINC01553","ANK3","LINC02059","MIR4280HG","TNIP1",
    "RASGEF1C","HLA-DRB1","HLA-DQA1","UNC5CL","OARD1",
    "TREM2","TREML2","CD2AP","HS3ST5","HDAC2-AS2","TPCN1",
    "FERMT2","SLC24A4","KLF16","SIGLEC11","RNU6-1307P",
    "LILRB2","RBCK1","CASS4","SLC2A4RG","APP","CYYR1",
    "ADAMTS1","DEDD","UFC1","QKILA","SETD7","NDUFAF6",
    "ATP8B4","TNFSF12","TNFSF13","ARHGAP45","SORT1",
    "TMEM106B","PRKD3","GPR141","NME8","BIN1","NIFKP9",
    "WDR12","INPP5D","MME","IDUA","LINC02498","MIR572",
    "RHOH","ANKH","TSPAN14","BLNK","PLEKHA1","SPI1",
    "MS4A4A","RNU6-560P","LINC02695","SORL1","UMAD1",
    "IGHG1","IGHG3","IGHV3-64","IGHV3-65","RN7SL354P",
    "SPPL2A","MINDY2-DT","APH1B","SNX1","CTSH","DOC2A",
    "BCKDK","IL34","RNA5SP431","MAF","PLCG2","LINC00917",
    "FENDRR","FAM157C","WDR81","SCIMP","ZNF594-DT",
    "MYO15A","GRN","WNT3","ABI3","TSPOAP1-AS1",
    "CYB561","PPIAP55","ABCA7"
]

gwas_genes = set(gwas_genes)

In [ ]:
def overlap_size(set_a, set_b):
    return len(set_a & set_b), set_a & set_b

def overlap_coefficient(set_a, set_b):
    if min(len(set_a), len(set_b)) == 0:
        return 0
    return len(set_a & set_b) / min(len(set_a), len(set_b))

In [ ]:
G_tf = nx.DiGraph()

for _, row in master_regulators.iterrows():
    G_tf.add_edge(
        row["tf"],
        row["target"],
        weight=row["mode"],
        tissue=row["tissue"]
    )
tf_edges = set(
    tuple(sorted([u, v])) for u, v in G_tf.edges()
)

In [ ]:
ppi


In [ ]:
ppi["edge"] = ppi.apply(
    lambda x: tuple(sorted([x[7], x[8]])),
    axis=1
)

ppi_weights = (
    ppi.groupby("edge")
       .size()
       .reset_index(name="weight")
)

ppi_genes = set(ppi_weights["edge"].explode())

G_ppi = nx.Graph()

for _, row in ppi_weights.iterrows():
    a, b = row["edge"]
    G_ppi.add_edge(a, b, weight=row["weight"])

ppi_edges = set(G_ppi.edges())

In [55]:
summary_tf_ppi = []

from collections import defaultdict

ppi_neighbors = defaultdict(set)
for a, b in G_ppi.edges():
    ppi_neighbors[a].add(b)
    ppi_neighbors[b].add(a)

for (tissue, tf), df_tf in master_regulators.groupby(["tissue", "tf"]):
    targets = set(df_tf["target"])

    regulon_genes = targets | {tf}
    tf_edges = {
        tuple(sorted([tf, tgt]))
        for tgt in targets
    }

    n_overlap, overlap_edges = overlap_size(tf_edges, ppi_edges)
    ov_coef = overlap_coefficient(tf_edges, ppi_edges)

    physical_edges = set()
    for gene in regulon_genes:
        for neigh in ppi_neighbors.get(gene, []):
            physical_edges.add(tuple(sorted([gene, neigh])))
    if n_overlap > 0:
        summary_tf_ppi.append({
            "Tissue": tissue,
            "TF": tf,
            "Regulon size": len(targets),
            "Overlap size": len(overlap_edges),
            "TF-target ppi_edges": n_overlap,
            "Overlap coefficient": ov_coef,

            # opcional: lista compacta
            "TF-target ppi_edges": ";".join(
                f"{a}-{b}" for a, b in sorted(overlap_edges)
            )
        })

summary_tf_ppi_df = pd.DataFrame(summary_tf_ppi)
summary_tf_ppi_df

summary_tf_ppi_df.to_csv("summary_mrtf_ppi.csv", index=False)


In [ ]:
master_regulators
mrs = set(master_regulators["tf"])
gwas_mrs = mrs.intersection(gwas_genes)
gwas_mrs

In [ ]:
tflink = pd.read_csv("TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv", sep = "\t")

In [ ]:
tflink.head()

In [49]:
master_regulators = pd.read_csv("regulons_TFs_masters.txt", sep="\t")

# selecionar apenas as colunas de interesse
master_edges = master_regulators[["tissue", "mode", "tf", "target"]].drop_duplicates()

# selecionar e renomear colunas do TFLink
tflink_edges = (
    tflink[["Name.TF", "Name.Target"]]
    .drop_duplicates()
    .rename(columns={"Name.TF": "tf", "Name.Target": "target"})
)

# interseção das arestas (TF → Target)
tflink_master_intersection = pd.merge(
    tflink_edges,
    master_edges,
    on=["tf", "target"],
    how="inner"
)

tflink_master_intersection.drop_duplicates()
tflink_master_intersection.to_csv("summary_mrtf_tflink.csv", index=False)